In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/Himanshu-1703/reddit-sentiment-analysis/refs/heads/main/data/reddit.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.sample()['clean_comment'].values

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df[df['clean_comment'].isna()]

In [ ]:
df[df['clean_comment'].isna()]['category'].value_counts()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated()]

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df[(df['clean_comment'].str.strip() == '')]

In [ ]:
df = df[~(df['clean_comment'].str.strip() == '')]

In [ ]:
df['clean_comment'] = df['clean_comment'].str.lower()

df.head()

In [ ]:
df[df['clean_comment'].apply(lambda x: x.endswith(' ') or x.startswith(' '))]

In [ ]:
df['clean_comment'] = df['clean_comment'].apply(lambda x: x.strip())

df['clean_comment'].apply(lambda x: x.endswith(' ') or x.startswith(' ')).sum()

In [ ]:
url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
comments_with_urls = df[df['clean_comment'].str.contains(url_pattern, regex=True)]

comments_with_urls.head()

In [ ]:
comments_with_newline = df[df['clean_comment'].str.contains('\n')]

comments_with_newline.head()

In [ ]:
df['clean_comment'] = df['clean_comment'].str.replace('\n', ' ', regex=True)

comments_with_newline_remaining = df[df['clean_comment'].str.contains('\n')]

comments_with_newline_remaining.head()

EDA

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(data=df, x='category')
plt.show()

In [ ]:
# frequency distribution of sentiments

df['category'].value_counts(normalize=True).mul(100).round(2)

In [ ]:
df['word_count'] = df['clean_comment'].apply(lambda x: len(x.split()))

In [ ]:
df.sample(5)

In [ ]:
df['word_count'].describe()

In [ ]:
sns.displot(df['word_count'], kde=True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.kdeplot(df[df['category'] == 1]['word_count'], label='Positive', fill=True)
sns.kdeplot(df[df['category'] == 0]['word_count'], label='Neutral', fill=True)
sns.kdeplot(df[df['category'] == -1]['word_count'], label='Negative', fill=True)

plt.title('Word Count Distribution by Category')
plt.xlabel('Word Count')
plt.ylabel('Density')

plt.legend()
plt.show()

In [ ]:
sns.boxplot(df['word_count'])

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='category', y='word_count')
plt.title('Boxplot of Word Count by Category')
plt.xlabel('Category')
plt.ylabel('Word Count')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='category', y='word_count', alpha=0.5)
plt.title('Scatterplot of Word Count bt Category')
plt.xlabel('Category')
plt.ylabel('Word Count')
plt.show()

In [ ]:
!pip install nltk

In [ ]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

df['num_stop_words'] = df['clean_comment'].apply(lambda x: len([word for word in x.split() if word in stop_words]))

df.head(5)

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['num_stop_words'], kde=True)
plt.title('Distribution of Stop Word Count in Comments')
plt.xlabel('Number of Stop Words')
plt.ylabel('Frequency')
plt.show()

In [ ]:
sns.barplot(df, x='category', y='num_stop_words', estimator='median')

In [ ]:
from collections import Counter

all_stop_words = [word for comment in df['clean_comment'] for word in comment.split() if word in stop_words]

most_common_stop_words = Counter(all_stop_words).most_common(25)

top_25_df = pd.DataFrame(most_common_stop_words, columns=['stop_word', 'count'])

plt.figure(figsize=(10, 6))
sns.barplot(data=top_25_df, x='count', y='stop_word', palette='viridis')
plt.title('Top 25 Most Common Stop Words')
plt.xlabel('Count')
plt.ylabel('Stop Word')
plt.show()

In [ ]:
df['num_chars'] = df['clean_comment'].apply(len)

df.head()

In [ ]:
df['num_chars'].describe()

In [ ]:
from collections import Counter

all_text = ' '.join(df['clean_comment'])

char_frequency = Counter(all_text)

char_frequency_df = pd.DataFrame(char_frequency.items(), columns=['character', 'frequency']).sort_values(by='frequency')

In [ ]:
char_frequency_df['character'].values

In [ ]:
char_frequency_df.tail(50)

In [ ]:
df['num_punctuation_chars'] = df['clean_comment'].apply(
    lambda x: sum([1 for char in x if char in '.,!?:;"\'()[]{}-'])
)

df.sample(5)

In [ ]:
df['num_punctuation_chars'].describe()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def get_top_ngrams(corpus, n=None):
  vec = CountVectorizer(ngram_range=(n, n), stop_words='english').fit(corpus)
  bag_of_words = vec.transform(corpus)
  sum_words = bag_of_words.sum(axis=0)
  words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
  word_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
  return word_freq[:n]

top_25_bigrams = get_top_ngrams(df['clean_comment'], 25)

top_25_bigrams_df = pd.DataFrame(top_25_bigrams, columns=['bigram', 'count'])

plt.figure(figsize=(12, 8))
sns.barplot(data=top_25_bigrams_df, x='count', y='bigram', palette='magma')
plt.title('Top 25 Most Common Bigrams')
plt.xlabel('Count')
plt.ylabel('Bigram')
plt.show()

In [ ]:
def get_top_trigrams(corpus, n=None):
  vec = CountVectorizer(ngram_range=(3, 3), stop_words='english').fit(corpus)
  bag_of_words = vec.transform(corpus)
  sum_words = bag_of_words.sum(axis=0)
  words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
  words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
  return words_freq[:n]

top_25_trigrams = get_top_trigrams(df['clean_comment'], 25)

top_25_trigrams_df = pd.DataFrame(top_25_trigrams, columns=['trigram', 'count'])

plt.figure(figsize=(12, 8))
sns.barplot(data=top_25_trigrams_df, x='count', y='trigram', palette='coolwarm')
plt.title('Top 25 Most Common Trigrams')
plt.xlabel('Count')
plt.ylabel('Trigram')
plt.show()

In [ ]:
import re

df['clean_comment'] = df['clean_comment'].apply(lambda x: re.sub(r'[^A-Za-z0-9\s!?.,]', '', str(x)))

In [ ]:
all_text = ' '.join(df['clean_comment'])

char_frequency = Counter(all_text)

char_frequency_df = pd.DataFrame(char_frequency.items(), columns=['character', 'frequency']).sort_values(by='frequency')

char_frequency_df

In [ ]:
df.head()

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english')) - {'not', 'but', 'however', 'no', 'yet'}

df['clean_comment'] = df['clean_comment'].apply(lambda x: ' '.join([word for word in x.split() if word.lower() not in stop_words]))

df.head()

In [ ]:
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

df['clean_comment'] = df['clean_comment'].apply(lambda x: ' '.join([lemmatizer.lemmatize(word) for word in x.split()]))

df.head()

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

def plot_word_cloud(text):
  wordcloud = WordCloud(width=800, height=400, background_color='white').generate(' '.join(text))
  plt.figure(figsize=(10, 5))
  plt.imshow(wordcloud, interpolation='bilinear')
  plt.axis('off')
  plt.show()

plot_word_cloud(df['clean_comment'])

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

def plot_word_cloud(text):
  wordcloud = WordCloud(width=800, height=400, background_color='white').generate(' '.join(text))
  plt.figure(figsize=(10,5))
  plt.imshow(wordcloud, interpolation='bilinear')
  plt.axis('off')
  plt.show()

plot_word_cloud(df[df['category'] == 1]['clean_comment'])

In [ ]:
def plot_top_n_words(df, n=20):

  words = ' '.join(df['clean_comment']).split()

  counter = Counter(words)
  most_common_words = counter.most_common(n)

  words, counts = zip(*most_common_words)

  plt.figure(figsize=(10, 6))
  sns.barplot(x=list(counts), y=list(words))
  plt.title(f'Top {n} Most Frequent Words')
  plt.xlabel('Frequency')
  plt.ylabel('Words')
  plt.show()

plot_top_n_words(df, n=30)

In [ ]:
def plot_top_n_words_by_category(df, n=20):

  word_category_counts = {}

  for idx, row in df.iterrows():
    words = row['clean_comment'].split()
    category = row['category']

    for word in words:
      if word not in word_category_counts:
        word_category_counts[word] = {-1: 0, 0: 0, 1: 0}
      word_category_counts[word][category] += 1

  total_word_counts = {word: sum(counts.values()) for word, counts in word_category_counts.items()}

  most_common_words = sorted(total_word_counts.items(), key=lambda x: x[1], reverse=True)[:n]
  top_words = [word for word, _ in most_common_words]

  word_labels = top_words
  negative_counts = [word_category_counts[word][-1] for word in top_words]
  neutral_counts = [word_category_counts[word][0] for word in top_words]
  positive_counts = [word_category_counts[word][1] for word in top_words]

  plt.figure(figsize=(12, 8))
  bar_width = 0.75

  plt.barh(word_labels, negative_counts, color='red', label='Negative', height=bar_width)
  plt.barh(word_labels, neutral_counts, left=negative_counts, color='yellow', label='Neutral', height=bar_width)
  plt.barh(word_labels, positive_counts, left=np.array(negative_counts) + np.array(neutral_counts), color='green', label='Positive', height=bar_width)
  plt.xlabel('Frequency')
  plt.ylabel('Words')
  plt.title(f'Top {n} Most Frequent Words by Category')
  plt.legend()
  plt.gca().invert_yaxis()
  plt.show()

plot_top_n_words_by_category(df, n=20)